# Silver – Olist
Opschonen van bronze-data: juiste datatypes, dubbele rijen eruit, ongeldige rijen eruit.


In [0]:
from pyspark.sql import functions as F

CATALOG = "bazar9_dbw"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"

def read_bronze(name):
    """Lees een bronze-tabel."""
    return spark.table(f"{BRONZE}.{name}")

def tc(col, datatype):
    """Zet tekst om naar een datatype. Lukt dat niet, dan wordt het leeg (NULL)."""
    return F.expr(f"try_cast({col} AS {datatype})")

def ts(col):
    """Zet tekst om naar datum + tijd."""
    return F.expr(f"try_to_timestamp({col})")

def save_silver(df, name):
    """Sla op als silver-tabel en toon het aantal rijen."""
    (
        df.withColumn("_processed_at", F.current_timestamp())
        .write.mode("overwrite")
        .option("overwriteSchema", True)
        .saveAsTable(f"{SILVER}.{name}")
    )
    print(f"{name}: {spark.table(f'{SILVER}.{name}').count():,} rijen")

In [0]:
orders = (
    read_bronze("orders")
    .select(
        "order_id",
        "customer_id",
        "order_status",
        ts("order_purchase_timestamp").alias("purchased_at"),
        ts("order_approved_at").alias("approved_at"),
        ts("order_delivered_carrier_date").alias("delivered_carrier_at"),
        ts("order_delivered_customer_date").alias("delivered_customer_at"),
        ts("order_estimated_delivery_date").alias("estimated_delivery_at"),
        "_ingested_at",
    )
    .filter(F.col("order_id").isNotNull())
    .dropDuplicates(["order_id"])
)
save_silver(orders, "orders")

In [0]:
order_items = (
    read_bronze("order_items")
    .select(
        "order_id",
        tc("order_item_id", "int").alias("order_item_id"),
        "product_id",
        "seller_id",
        ts("shipping_limit_date").alias("shipping_limit_at"),
        tc("price", "decimal(10,2)").alias("price"),
        tc("freight_value", "decimal(10,2)").alias("freight_value"),
        "_ingested_at",
    )
    .dropDuplicates(["order_id", "order_item_id"])
)
save_silver(order_items, "order_items")

In [0]:
customers = (
    read_bronze("customers")
    .select(
        "customer_id",
        "customer_unique_id",
        F.col("customer_zip_code_prefix").alias("zip_code_prefix"),
        F.initcap(F.trim("customer_city")).alias("city"),
        F.upper(F.trim("customer_state")).alias("state"),
        "_ingested_at",
    )
    .dropDuplicates(["customer_id"])
)
save_silver(customers, "customers")

sellers = (
    read_bronze("sellers")
    .select(
        "seller_id",
        F.col("seller_zip_code_prefix").alias("zip_code_prefix"),
        F.initcap(F.trim("seller_city")).alias("city"),
        F.upper(F.trim("seller_state")).alias("state"),
        "_ingested_at",
    )
    .dropDuplicates(["seller_id"])
)
save_silver(sellers, "sellers")

In [0]:
geolocation = (
    read_bronze("geolocation")
    .groupBy(F.col("geolocation_zip_code_prefix").alias("zip_code_prefix"))
    .agg(
        F.avg(tc("geolocation_lat", "double")).alias("lat"),
        F.avg(tc("geolocation_lng", "double")).alias("lng"),
        F.first(F.upper("geolocation_state")).alias("state"),
        F.max("_ingested_at").alias("_ingested_at"),
    )
)
save_silver(geolocation, "geolocation")

In [0]:
translation = read_bronze("product_category_translation").select(
    "product_category_name",
    F.col("product_category_name_english").alias("category_en"),
)

products = (
    read_bronze("products")
    .join(translation, "product_category_name", "left")
    .select(
        "product_id",
        F.coalesce("category_en", "product_category_name", F.lit("unknown")).alias("category"),
        tc("product_name_lenght", "int").alias("name_length"),
        tc("product_description_lenght", "int").alias("description_length"),
        tc("product_photos_qty", "int").alias("photos_qty"),
        tc("product_weight_g", "int").alias("weight_g"),
        tc("product_length_cm", "int").alias("length_cm"),
        tc("product_height_cm", "int").alias("height_cm"),
        tc("product_width_cm", "int").alias("width_cm"),
        "_ingested_at",
    )
    .dropDuplicates(["product_id"])
)
save_silver(products, "products")

In [0]:
payments = (
    read_bronze("order_payments")
    .select(
        "order_id",
        tc("payment_sequential", "int").alias("payment_sequential"),
        "payment_type",
        tc("payment_installments", "int").alias("installments"),
        tc("payment_value", "decimal(10,2)").alias("payment_value"),
        "_ingested_at",
    )
    .dropDuplicates(["order_id", "payment_sequential"])
)
save_silver(payments, "order_payments")

reviews = (
    read_bronze("order_reviews")
    .select(
        "review_id",
        "order_id",
        tc("review_score", "int").alias("review_score"),
        F.col("review_comment_title").alias("comment_title"),
        F.col("review_comment_message").alias("comment_message"),
        ts("review_creation_date").alias("created_at"),
        ts("review_answer_timestamp").alias("answered_at"),
        "_ingested_at",
    )
    .filter(F.col("review_score").between(1, 5))
    .dropDuplicates(["review_id", "order_id"])
)
save_silver(reviews, "order_reviews")

In [0]:
%sql
SELECT 'orders zonder klant' AS controle, COUNT(*) AS aantal
FROM bazar9_dbw.silver.orders o
LEFT ANTI JOIN bazar9_dbw.silver.customers c ON o.customer_id = c.customer_id

UNION ALL
SELECT 'orderregels zonder order', COUNT(*)
FROM bazar9_dbw.silver.order_items i
LEFT ANTI JOIN bazar9_dbw.silver.orders o ON i.order_id = o.order_id

UNION ALL
SELECT 'orderregels zonder product', COUNT(*)
FROM bazar9_dbw.silver.order_items i
LEFT ANTI JOIN bazar9_dbw.silver.products p ON i.product_id = p.product_id

UNION ALL
SELECT 'betalingen zonder order', COUNT(*)
FROM bazar9_dbw.silver.order_payments p
LEFT ANTI JOIN bazar9_dbw.silver.orders o ON p.order_id = o.order_id

UNION ALL
SELECT 'negatieve prijzen', COUNT(*)
FROM bazar9_dbw.silver.order_items
WHERE price < 0

UNION ALL
SELECT 'orders zonder aankoopdatum', COUNT(*)
FROM bazar9_dbw.silver.orders
WHERE purchased_at IS NULL;